# Tarea 1
__Curso:__ Tópicos avanzados en Inteligencia Artificial 1

__Programa:__ MIA 2025

__Profesor:__ Anthony D. Cho

__Ayudante corrector:__ Luis Oliveros

## Instrucciones
* La actividad debe ser realizada por los grupos capstones
* Por favor responder en este mismo notebook (una entrega por grupo)
* Renombrar el archivo agregando el apellido de las y los integrantes, por ejemplo actividad1_Tupper_Tudor_Gorosito_Acosta.ipynb
* Subir el archivo al link de entrega Actividad 1 en webcursos que será habilitado

__Fecha de entrega:__ Fecha límite de entrega 28 de junio de 2026.

__Integrantes:__ (RUT, Nombre y Apellido)

*
*
*

## Librerias

In [ ]:
import os
import matplotlib.pyplot as plt
from numpy import linspace, mean, argmin
from pandas import read_csv, DataFrame

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

## Keras - tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers

## Agregar las otras librerias que necesiten


## Funciones personalizadas

In [ ]:
def plot_history(history, width=12, height=6):
  """
  DESCRIPTION:
    History performance of the keras model

  INPUT:
    @param history: history of performance of fitted model
    @type history: tensorflow.python.keras.callbacks.History

  OUTPUT:
    A graphic
  """

  ## Metrics keys stored in tensorflow object
  keys = list(history.history.keys())

  ## Number of epoch used for fit the model
  epoch = range(1, len(history.epoch) +1)

  ## Check if validation set was used.
  withValidation = False
  for key in keys:
    if 'val' in key:
      withValidation = True

  ## Number of metrics
  nMetrics = len(keys)
  if withValidation:
    nMetrics = nMetrics//2

  ## Plot-space instance
  plt.figure(figsize=(width, height))

  for i in range(nMetrics):
    plt.subplot(nMetrics, 1, i+1)

    ## Plot (train) metric value
    labelMetric = keys[i]
    metric = history.history[keys[i]]
    plt.plot(epoch, metric, 'o-', label=labelMetric)

    if withValidation:
      ## Plot (validation) metric value
      labelMetricVal = keys[i+nMetrics]
      metricVal = history.history[keys[i+nMetrics]]
      plt.plot(epoch, metricVal, 'o-', label=labelMetricVal)

    plt.xlim(epoch[0], epoch[-1])
    plt.legend()
    plt.grid()

  plt.xlabel('Epoch')
  plt.show()

## Dataset

<center>
    <img src=https://www.xenonstack.com/hs-fs/hubfs/xenonstack-credit-card-fraud-detection.png?width=1920&height=1080&name=xenonstack-credit-card-fraud-detection.png width=800>
</center>

El conjunto de datos contiene transacciones realizadas con tarjetas de crédito en septiembre de 2013 en usuarios europeos.
Este conjunto de datos presenta transacciones que ocurrieron en dos días, donde tiene diagnosticado 492 casos fraudes de 284.807 transacciones. El conjunto de datos está muy desequilibrado, la clase positiva (fraudes) representa el 0,172% de todas las transacciones.

* La data y detalles está completamente disponible en [Kaggle: Credit Card Fraud Detection](https://www.kaggle.com/mlg-ulb/creditcardfraud)

<p style="color:red"> <b>Instrucciones para usuarios que usan Windows: </b></p>

* Descargar la data pulsando [¡¡AQUI!!](https://github.com/adoc-box/Datasets/raw/main/creditcard.zip)
* Descomprime el archivo __creditcard.zip__. Luego, dejar el archivo __'creditcard.csv__ junto con el script.
* Ya con eso, debería poder ejecutar el script sin ningun problema.

In [ ]:
dataset_name = 'creditcard.csv'

## Solo para Linux y MacOS
if os.name == 'posix':
    if not os.path.exists("creditcard.zip"):
        ## Download files

        !wget https://github.com/adoc-box/Datasets/raw/main/creditcard.zip

        ## un-compress joined zip file
        !unzip creditcard.zip

    else:
        if os.path.exists(dataset_name):
            !rm creditcard.csv
        !unzip creditcard.zip

else:
    print('Seguir las instrucciones para usuarios de Windows mencionado arriba.')

In [ ]:
## Load data
data = read_csv('creditcard.csv')

print("No. of unique labels ", len(data['Class'].unique()))
print("Label values ",data.Class.unique())
# 0 is for normal credit card transaction
# 1 is for fraudulent credit card transaction

print('-------')
print("Break down of the Normal and Fraud Transactions")
print( data['Class'].value_counts(sort = True), end=2*'\n' )

data.head()

In [ ]:
## Data splitting: the normal and fradulent transactions in separate dataframe
normal_data = data.query("Class == 0")
fraud_data = data.query("Class == 1")

## Visualize transaction amounts for normal and fraudulent transactions
bins = linspace(200, 2500, 100)
plt.figure(figsize=(14, 4))
plt.hist(normal_data.Amount, bins=bins, alpha=1, density=True, label='Normal')
plt.hist(fraud_data.Amount, bins=bins, alpha=0.5, density=True, label='Fraud')
plt.legend(loc='upper right')
plt.title("Transaction amount vs Percentage of transactions")
plt.xlabel("Transaction amount (USD)"); plt.ylabel("Percentage of transactions");
plt.tight_layout()
plt.show()

In [ ]:
data.describe(include='all')

### Data pre-processing

In [ ]:
## predictors and labels assigments
X, Y = data.drop(columns=['Time', 'Class']), data['Class']

## Train, validate and test set
X_trainVal, X_test, y_trainVal, y_test = train_test_split(X, Y, stratify=Y,
                                                          test_size=0.2,
                                                          random_state=84)

X_train, X_val, y_train, y_val = train_test_split(X_trainVal, y_trainVal,
                                                  stratify=y_trainVal,
                                                  test_size=0.2,
                                                  random_state=84)

print('Train (Shape) - X: {}, y: {}'.format(X_train.shape, y_train.shape))
print('Val (Shape) - X: {}, y: {}'.format(X_val.shape, y_val.shape))
print('Test (Shape) - X: {}, y: {}\n'.format(X_test.shape, y_test.shape))


## Extract normal (no fraud) data for training and validate
X_train_normal = X_train[ y_train.values == 0 ]
X_val_normal = X_val[ y_val.values == 0 ]
X_trainVal_normal = X_trainVal[ y_trainVal.values == 0 ]

print('Train (Shape) Normal cases - X: {}'.format(X_train_normal.shape))
print('Val (Shape) Normal cases - X: {}'.format(X_val_normal.shape))
print('Train+Val (Shape) Normal cases - X: {}'.format(X_trainVal_normal.shape))



## Data normalization
scaler = MinMaxScaler().fit(X_train_normal)
X_train_normal = scaler.transform(X_train_normal)
X_val_normal = scaler.transform(X_val_normal)
X_trainVal_normal = scaler.transform(X_trainVal_normal)

X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

## Paso 1 (5 puntos):

Utilice el conjunto de entrenamiento y validación de la mejor forma tal de poder definir los hiperparámetros (número de capas ocultas, número de unidades en las capas, funciones de activación, número de épocas) de un modelo **Autoencoder Variacional (VAE)**  para abordar el problema de detección de fraudes siguiendo un enfoque similar al ejemplo de Vanilla Autoencoder visto en clases.

Deje las 3 mejores cofiguraciones probados reportando también las métricas MAE y MSE de los conjuntos de entrenamiento y validación. También indiquen con cuál de las 3 configuraciones eligirian como su mejor opción.

## Paso 2 (3 puntos):

Con el mejor modelo entrenado en el Paso 1, encuentre el mejor umbral para el error tal que pueda maximizar la métrica F1-Score en el conjunto de validación.

## Paso 3 (2 puntos):

Usando el modelo entrenado y el umbral encontrado en los pasos anteriores, evaluar el modelo con el conjunto de prueba y entregue la matriz de confusión y el __classification_report__.